In [ ]:
import pandas as pd
import glob
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Customize high-resolution display for charts
%config InlineBackend.figure_format = 'retina'

# 1. Find analysis files
csv_files = glob.glob("report_metrics_*.csv")
if not csv_files:
    print("No CSV files found starting with 'report_metrics_'")
else:
    print(f"Found {len(csv_files)} files: {csv_files}")

    # 2. Merge data
    dfs = [pd.read_csv(f) for f in csv_files]
    df_all = pd.concat(dfs, ignore_index=True)

    # 3. Preprocessing
    if df_all['Accuracy'].max() <= 1.0:
        df_all['Accuracy'] = df_all['Accuracy'] * 100

    num_rounds_per_task = df_all['Round'].max()
    df_all['Global_Round'] = df_all['Train_Task'] * num_rounds_per_task + df_all['Round']

    print(f"Loaded {df_all.shape[0]} rows of data. Generating charts...")

    # 4. Plot charts
    sns.set_theme(style="whitegrid")
    eval_tasks = sorted(df_all['Eval_Task'].unique())
    num_eval_tasks = len(eval_tasks)

    fig, axes = plt.subplots(nrows=num_eval_tasks, ncols=1, 
                             figsize=(12, 5 * num_eval_tasks), 
                             sharex=True)

    if num_eval_tasks == 1:
        axes = [axes]

    for ax, eval_id in zip(axes, eval_tasks):
        subset = df_all[df_all['Eval_Task'] == eval_id]
        
        sns.lineplot(
            data=subset,
            x='Global_Round',
            y='Accuracy',
            hue='Method',
            linewidth=2.5,
            ax=ax,
            palette="tab10"
        )
        
        ax.set_title(f"Accuracy on Test Set of Task {eval_id}", fontweight='bold', fontsize=14)
        ax.set_ylabel("Accuracy (%)", fontsize=12)
        ax.set_ylim(-5, 105)

        # Draw separator lines for Tasks
        for t_split in df_all['Train_Task'].unique():
            if t_split > 0:
                split_round = t_split * num_rounds_per_task
                ax.axvline(x=split_round, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
                ax.text(split_round + 1, 5, f'Start Task {t_split}', color='red', fontsize=10, rotation=90)
                
        ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', title="Strategies")

    axes[-1].set_xlabel("Global Rounds", fontsize=12)
    plt.tight_layout()

    # Display the chart directly in the notebook
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import glob

# ==========================================
# 1. ORIGINAL PAPER BASELINES (Hardcoded)
# ==========================================
PAPER_BASELINES = {
    'CIFAR10': {
        0.5: {'Finetune': 38.71, '+GDR': 62.13, '+TTS': 41.34, '+GDR+TTS': 64.11},
        1.0: {'Finetune': 40.49, '+GDR': 63.81, '+TTS': 42.55, '+GDR+TTS': 65.20}
    },
    'CIFAR100': {
        0.1: {'Finetune': 15.17, '+GDR': 45.28, '+TTS': 17.32, '+GDR+TTS': 46.40},
        0.5: {'Finetune': 16.75, '+GDR': 47.66, '+TTS': 17.14, '+GDR+TTS': 49.76},
        1.0: {'Finetune': 17.15, '+GDR': 51.47, '+TTS': 19.32, '+GDR+TTS': 52.06}
    },
    'TinyImageNet': {
        0.1: {'Finetune': 6.06, '+GDR': 17.24, '+TTS': 6.67, '+GDR+TTS': 18.37},
        0.5: {'Finetune': 6.00, '+GDR': 17.89, '+TTS': 6.92, '+GDR+TTS': 18.86},
        1.0: {'Finetune': 6.40, '+GDR': 18.04, '+TTS': 7.27, '+GDR+TTS': 18.78}
    }
}

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def extract_final_average_accuracy(csv_path):
    """Reads the CSV log and returns the average accuracy of the final round."""
    if not os.path.exists(csv_path):
        print(f"  [!] Warning: File not found -> {csv_path}. Returning 0.0")
        return 0.0
    
    try:
        df = pd.read_csv(csv_path)
        final_train_task = df['Train_Task'].max()
        final_round = df[df['Train_Task'] == final_train_task]['Round'].max()
        final_results = df[(df['Train_Task'] == final_train_task) & (df['Round'] == final_round)]
        avg_acc = final_results['Accuracy'].mean()
        return round(avg_acc, 2)
    except Exception as e:
        print(f"  [!] Error reading {csv_path}: {e}")
        return 0.0

def get_experiment_folder(base_dir, dataset_name, beta_value):
    """Automatically finds the correct folder for the given dataset and beta."""
    # Convert dataset name to lowercase to match folder names (e.g., CIFAR10 -> cifar10)
    dataset_folder = os.path.join(base_dir, dataset_name.lower())
    if not os.path.exists(dataset_folder):
        print(f"Dataset directory not found: {dataset_folder}")
        return None
    
    # Search for a folder that ends with the exact beta value
    search_pattern = os.path.join(dataset_folder, f"*-{beta_value}")
    matching_folders = glob.glob(search_pattern)
    
    # Filter only directories
    matching_folders = [f for f in matching_folders if os.path.isdir(f)]
    
    if matching_folders:
        return matching_folders[0]  # Return the first matching directory
    else:
        print(f"Experiment folder not found for Beta={beta_value} inside {dataset_folder}")
        return None

def plot_reproduction_comparison(dataset_name, beta_value, base_dir="report"):
    """Plots a grouped bar chart dynamically reading from the directory structure."""
    methods = ['Finetune', '+GDR', '+TTS', '+GDR+TTS']
    
    # Standardized file names based on your consistency fix
    file_names = {
        'Finetune': 'report_metrics_none.csv',
        '+GDR': 'report_metrics_gdr.csv',
        '+TTS': 'report_metrics_random_tts.csv',
        '+GDR+TTS': 'report_metrics_gdr_tts.csv'
    }
    
    # 1. Fetch Paper Baseline Data
    try:
        paper_dict = PAPER_BASELINES[dataset_name][beta_value]
        paper_scores = [paper_dict[m] for m in methods]
    except KeyError:
        print(f"Error: No paper baseline data found for {dataset_name} (Beta={beta_value})")
        return

    # 2. Find the correct directory
    exp_folder = get_experiment_folder(base_dir, dataset_name, beta_value)
    if not exp_folder:
        return

    print(f"Reading data from: {exp_folder}")

    # 3. Read Data
    repro_scores = []
    for method in methods:
        csv_file_path = os.path.join(exp_folder, file_names[method])
        acc = extract_final_average_accuracy(csv_file_path)
        repro_scores.append(acc)

    # 4. Plotting Setup
    x = np.arange(len(methods))
    width = 0.35  
    
    fig, ax = plt.subplots(figsize=(10, 6))
    rects1 = ax.bar(x - width/2, paper_scores, width, label='Original Paper', color='#4C72B0')
    rects2 = ax.bar(x + width/2, repro_scores, width, label='Our Reproduction', color='#DD8452')
    
    # Customization
    ax.set_ylabel('Average Accuracy (%)', fontsize=12, fontweight='bold')
    ax.set_title(f'Performance Comparison: Paper vs Reproduction\nDataset: {dataset_name} | Dirichlet $\\beta$ = {beta_value}', 
                 fontsize=14, fontweight='bold', pad=15)
    ax.set_xticks(x)
    ax.set_xticklabels(methods, fontsize=12)
    ax.set_ylim(0, max(max(paper_scores), max(repro_scores)) + 15) # Add space for text
    
    ax.legend(fontsize=11, loc='upper left')
    ax.yaxis.grid(True, linestyle='--', alpha=0.7)
    ax.set_axisbelow(True)
    
    # Value Annotations
    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height}%',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3), 
                        textcoords="offset points",
                        ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    autolabel(rects1)
    autolabel(rects2)
    
    plt.tight_layout()
    plt.show()

# ==========================================
# 3. EXECUTION BLOCK
# ==========================================
# Make sure your notebook is in the same folder that contains the 'report/' directory.

# Choose your dataset and beta here:
TARGET_DATASET = 'CIFAR10'   # Must exactly match the keys in PAPER_BASELINES (e.g., 'CIFAR10', 'CIFAR100')
TARGET_BETA = 0.5            # 0.1, 0.5, or 1.0

# plot_reproduction_comparison(
#     dataset_name=TARGET_DATASET, 
#     beta_value=TARGET_BETA, 
#     base_dir="report"        # The root folder of your logs
# )

# You can easily loop through everything you have:
for b in [0.1, 0.5, 1.0]:
   plot_reproduction_comparison('TinyImageNet', b)

In [ ]:
import pandas as pd
import numpy as np
import glob
import re
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# --- DISPLAY SETTINGS ---
%config InlineBackend.figure_format = 'retina'
sns.set_theme(style="white")
plt.close('all')

# DEFINE ALGORITHM PRIORITY (CUSTOM ORDER)
METHOD_PRIORITY = {
    'NONE': 0,
    'RANDOM TTS': 1,
    'GDR': 2,
    'GDR TTS': 3,
    'KMEANS TTS': 4,
    'SVD KMEANS TTS': 5
}

csv_files = glob.glob("report_metrics*.csv")
results = []

for file in csv_files:
    match_tasks = re.search(r'_t(\d+)', file)
    match_alpha = re.search(r'_a(\d+(?:\.\d+)?)', file)
    
    num_tasks = int(match_tasks.group(1)) if match_tasks else 10
    alpha_str = match_alpha.group(1) if match_alpha else "0.5"
    alpha = float(alpha_str)
    
    method_raw = file.replace("report_metrics_", "").replace(".csv", "")
    if match_tasks: method_raw = method_raw.replace(f"t{num_tasks}", "")
    if match_alpha: method_raw = method_raw.replace(f"a{alpha_str}", "")
    
    method_clean = method_raw.replace('_tabular', '').replace('_', ' ').replace('+', ' ').strip().upper()
    if not method_clean:
        method_clean = "NONE"
    
    try:
        df = pd.read_csv(file)
        if not df.empty:
            final_train_task = df['Train_Task'].max()
            final_round = df[df['Train_Task'] == final_train_task]['Round'].max()
            final_results = df[(df['Train_Task'] == final_train_task) & (df['Round'] == final_round)]
            
            for _, row in final_results.iterrows():
                eval_task = int(row['Eval_Task'])
                acc = row['Accuracy']
                if acc <= 1.0: acc *= 100
                
                method_rank = METHOD_PRIORITY.get(method_clean, 99)
                results.append({
                    'Tasks_Group': num_tasks,  
                    'Method': method_clean,
                    'Method_Rank': method_rank,
                    'Alpha': alpha,
                    'Config': f"[Tasks: {num_tasks}] {method_clean} | α={alpha}", 
                    'Eval_Task_Num': eval_task,
                    'Accuracy': acc
                })
    except Exception as e:
        print(f"Error reading {file}: {e}")

df_results = pd.DataFrame(results)

if df_results.empty:
    print("No data found!")
else:
    print(f"Generating Stacked Memory Retention Heatmap (Including Average & Stability)")
    
    # 1. Sort Y-axis order (Tasks -> Algorithm Rank -> Alpha)
    unique_configs = df_results[['Tasks_Group', 'Method', 'Method_Rank', 'Alpha', 'Config']].drop_duplicates()
    unique_configs = unique_configs.sort_values(by=['Tasks_Group', 'Method_Rank', 'Alpha'])
    df_results['Config'] = pd.Categorical(df_results['Config'], categories=unique_configs['Config'], ordered=True)
    
    # 2. Pivot to create Matrix table
    pivot_df = df_results.pivot_table(index='Config', columns='Eval_Task_Num', values='Accuracy', aggfunc='max')
    pivot_df = pivot_df[sorted(pivot_df.columns)]
    
    # Save the list of Task columns before adding summary columns
    task_cols = [f'Task {c}' for c in pivot_df.columns]
    pivot_df.columns = task_cols
    
    # ADD AVERAGE AND STABILITY (STANDARD DEVIATION) COLUMNS
    pivot_df['Average'] = pivot_df[task_cols].mean(axis=1)
    pivot_df['Stability (Std)'] = pivot_df[task_cols].std(axis=1).fillna(0)

    # 3. Create Custom Annotation Matrix
    mask = pivot_df.isna()
    annot_matrix = np.empty(pivot_df.shape, dtype=object)
    
    max_avg = pivot_df['Average'].max() 
    
    # Find the most stable model (Lowest Std but Average score must be good enough to avoid selecting 0% models)
    valid_models = pivot_df[pivot_df['Average'] > 30]
    min_std = valid_models['Stability (Std)'].min() if not valid_models.empty else pivot_df['Stability (Std)'].min()
    
    for i in range(pivot_df.shape[0]):
        for j in range(pivot_df.shape[1]):
            val = pivot_df.iloc[i, j]
            col_name = pivot_df.columns[j]
            
            if pd.isna(val):
                annot_matrix[i, j] = ""
            else:
                formatted_val = f"{val:.1f}"
                # Mark Best for highest Average
                if col_name == 'Average' and val == max_avg:
                    annot_matrix[i, j] = f"Best: {formatted_val}"
                # Mark Top for lowest standard deviation
                elif col_name == 'Stability (Std)' and val == min_std:
                    annot_matrix[i, j] = f"Top: {formatted_val}"
                # Add ± effect to Std column cells for formatting
                elif col_name == 'Stability (Std)':
                    annot_matrix[i, j] = f"± {formatted_val}"
                else:
                    annot_matrix[i, j] = formatted_val

    # 4. Draw on the aggregate board
    # Increase horizontal size to accommodate extra columns
    plt.figure(figsize=(max(15, len(pivot_df.columns)*1.2), len(pivot_df)*0.4 + 2))
    cmap_custom = sns.diverging_palette(250, 20, as_cmap=True) 

    ax = sns.heatmap(pivot_df, 
                mask=mask,           
                annot=annot_matrix,      
                fmt="",                 
                cmap=cmap_custom, 
                vmin=0, vmax=100,
                linewidths=1.5,
                linecolor='white',
                cbar_kws={'label': 'Metrics Value Indicator', 'shrink': 0.8},
                annot_kws={"fontsize": 11, "fontweight": "bold"})
                
    # Draw red line separating Tasks data area from Summary Dashboard (Average, Stability)
    summary_start_idx = len(task_cols)
    ax.axvline(summary_start_idx, color='maroon', linewidth=4, linestyle='--')
    # Draw light gray secondary line between Average and Stability columns
    ax.axvline(summary_start_idx + 1, color='gray', linewidth=2, linestyle=':')
                
    # DRAW HORIZONTAL BOUNDARIES BY LEVEL
    method_counts = unique_configs.groupby(['Tasks_Group', 'Method_Rank'], sort=False).size()
    method_lines = method_counts.cumsum().values[:-1]
    
    task_counts = unique_configs.groupby('Tasks_Group', sort=False).size()
    task_lines = task_counts.cumsum().values[:-1]
    
    for y_idx in method_lines:
        if y_idx in task_lines:
            ax.axhline(y_idx, color='black', linewidth=4.5) 
        else:
            ax.axhline(y_idx, color='black', linewidth=1.5, linestyle='solid') 
        
    plt.title("Continual Learning Memory Matrix\n(Highest Avg. Accuracy | Most Stable/Least Forgetting Model)", 
              fontsize=18, fontweight='bold', pad=15)
    plt.ylabel("Curriculum Strategy Timeline", fontsize=14, fontweight='bold')
    plt.xlabel("Accuracy per Evaluated Task & Stability Metrics", fontsize=14, fontweight='bold')
    
    plt.xticks(rotation=45, fontsize=11, fontweight='bold')
    plt.yticks(rotation=0, fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.show()